# Deepfake Voice Detection — Report Generation
### Step 3 of 4: Load model, analyze audio, and generate PDF report with charts
---

## Step 1 — Import Libraries

In [17]:
import subprocess
for pkg in ['librosa', 'shap', 'reportlab', 'noisereduce']:
    subprocess.run(['pip', 'install', pkg], capture_output=True)
    print(f'{pkg} ready!')
print('All libraries ready!')

librosa ready!
shap ready!
reportlab ready!
noisereduce ready!
All libraries ready!


In [18]:
import os
import numpy as np
import librosa
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pickle
import shap
from datetime import datetime
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                Image, Table, TableStyle, HRFlowable)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER, TA_LEFT
from reportlab.platypus import KeepTogether

print('Libraries imported successfully!')

Libraries imported successfully!


## Step 2 — Load Paths

In [19]:
# AUTO PATH DETECTION — works on any machine
BASE_DIR           = os.path.dirname(os.path.abspath('__file__'))
REAL_CHUNKS_FOLDER = os.path.join(BASE_DIR, 'dataset', 'real_chunks')
FAKE_CHUNKS_FOLDER = os.path.join(BASE_DIR, 'dataset', 'fake_chunks')
MODELS_DIR         = os.path.join(BASE_DIR, 'models')
REPORTS_FOLDER     = os.path.join(BASE_DIR, 'audio_reports')
CHARTS_PATH        = os.path.join(BASE_DIR, 'audio_reports', 'charts')

os.makedirs(REPORTS_FOLDER, exist_ok=True)
os.makedirs(CHARTS_PATH,    exist_ok=True)

MODEL_PATH      = os.path.join(MODELS_DIR, 'model.pkl')
SCALER_PATH     = os.path.join(MODELS_DIR, 'scaler.pkl')
REAL_MEANS_PATH = os.path.join(MODELS_DIR, 'real_means.npy')
REAL_STDS_PATH  = os.path.join(MODELS_DIR, 'real_stds.npy')

print(f'Project directory : {BASE_DIR}')
print(f'Models directory  : {MODELS_DIR}')
print(f'Reports directory : {REPORTS_FOLDER}')

Project directory : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection
Models directory  : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection\models
Reports directory : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection\audio_reports


## Step 3 — Load Model and Scaler from models/ Folder

In [20]:
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)

with open(SCALER_PATH, 'rb') as f:
    scaler = pickle.load(f)

real_means = np.load(REAL_MEANS_PATH)
real_stds  = np.load(REAL_STDS_PATH)

print(f'Model loaded  : {MODEL_PATH}')
print(f'Scaler loaded : {SCALER_PATH}')
print(f'Real means    : {len(real_means)} features loaded')
print(f'Real stds     : {len(real_stds)} features loaded')
print('Model and scaler ready!')

Model loaded  : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection\models\model.pkl
Scaler loaded : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection\models\scaler.pkl
Real means    : 40 features loaded
Real stds     : 40 features loaded
Model and scaler ready!


## Step 4 — Load Chunk Files from Dataset

In [21]:
real_chunk_files = sorted([f for f in os.listdir(REAL_CHUNKS_FOLDER) if f.endswith('.wav')])
fake_chunk_files = sorted([f for f in os.listdir(FAKE_CHUNKS_FOLDER) if f.endswith('.wav')])

print(f'Real chunk files : {len(real_chunk_files)}')
print(f'Fake chunk files : {len(fake_chunk_files)}')
print(f'Total chunks     : {len(real_chunk_files) + len(fake_chunk_files)}')

Real chunk files : 536
Fake chunk files : 327
Total chunks     : 863


## Step 5 — MFCC Human Readable Names and Color Constants

In [22]:
MFCC_NAMES = {
    'MFCC_1' : 'Overall Energy Level',
    'MFCC_2' : 'Spectral Shape (Brightness)',
    'MFCC_3' : 'Voice Texture Coarseness',
    'MFCC_4' : 'Vocal Tract Resonance',
    'MFCC_5' : 'Nasal Resonance Pattern',
    'MFCC_6' : 'Mid Frequency Tone',
    'MFCC_7' : 'Articulation Clarity',
    'MFCC_8' : 'Pitch Variation Pattern',
    'MFCC_9' : 'Voice Smoothness',
    'MFCC_10': 'Spectral Fine Structure',
    'MFCC_11': 'High Frequency Detail',
    'MFCC_12': 'Breathiness Level',
    'MFCC_13': 'Consonant Sharpness',
    'MFCC_14': 'Vowel Quality',
    'MFCC_15': 'Rhythm Pattern',
    'MFCC_16': 'Stress Pattern',
    'MFCC_17': 'Speaking Rate Variation',
    'MFCC_18': 'Voice Onset Pattern',
    'MFCC_19': 'Intonation Contour',
    'MFCC_20': 'Micro Pitch Fluctuation',
    'MFCC_21': 'Spectral Tilt',
    'MFCC_22': 'Formant Transition Speed',
    'MFCC_23': 'Voice Naturalness',
    'MFCC_24': 'Glottal Pulse Pattern',
    'MFCC_25': 'Subglottal Resonance',
    'MFCC_26': 'Palatal Resonance',
    'MFCC_27': 'Velar Resonance',
    'MFCC_28': 'Labial Sound Pattern',
    'MFCC_29': 'Dental Sound Texture',
    'MFCC_30': 'Fricative Pattern',
    'MFCC_31': 'Plosive Burst Pattern',
    'MFCC_32': 'Nasal Murmur Detail',
    'MFCC_33': 'Liquid Sound Texture',
    'MFCC_34': 'Glide Transition Pattern',
    'MFCC_35': 'Voice Tremor Detail',
    'MFCC_36': 'Micro Rhythm Pattern',
    'MFCC_37': 'Spectral Envelope Shape',
    'MFCC_38': 'Harmonic Structure',
    'MFCC_39': 'Noise Floor Pattern',
    'MFCC_40': 'Ultra Fine Voice Texture',
}

FAKE_RED    = '#E24B4A'
FAKE_LIGHT  = '#FCEBEB'
REAL_GREEN  = '#639922'
REAL_LIGHT  = '#EAF3DE'
NORMAL_BLUE = '#85B7EB'
GRAY        = '#888780'
DARK_GRAY   = '#444441'

print(f'MFCC names dictionary loaded — {len(MFCC_NAMES)} features named.')
print('Color constants defined.')

MFCC names dictionary loaded — 40 features named.
Color constants defined.


## Step 6 — predict_audio() Function
Analyzes any audio file by splitting it into 3-second chunks and predicting each one.

In [23]:
def predict_audio(Test_FILE):
    '''
    Predict whether an audio file is Real or Fake.
    Pipeline:
        1. Load audio
        2. Clean audio (noise reduction + spike removal)
        3. Split into 3 second chunks
        4. Extract MFCC features from each chunk
        5. Normalize using saved scaler
        6. Predict using saved model
        7. Return results with majority vote
    '''
    print(f'{"="*50}')
    print(f'Analyzing : {os.path.basename(Test_FILE)}')
    print(f'{"="*50}')

    # ── Step 1 — Load audio ───────────────────────────────
    try:
        audio, sr = librosa.load(Test_FILE, sr=16000, mono=True)
    except Exception as e:
        print(f'Could not load file: {e}')
        return None, None, None

    print(f'Duration     : {len(audio)/sr:.1f} seconds')

    # ── Step 2 — Clean audio ──────────────────────────────
    try:
        import noisereduce as nr
        from scipy.ndimage import gaussian_filter1d

        # Noise reduction using first 0.3 seconds as noise sample
        noise_sample  = audio[:int(0.3 * sr)]
        audio_cleaned = nr.reduce_noise(
            y             = audio,
            sr            = sr,
            y_noise       = noise_sample,
            prop_decrease = 0.6
        )

        # Spike removal
        frame_length = 512
        hop_length   = 256
        rms          = librosa.feature.rms(
            y            = audio_cleaned,
            frame_length = frame_length,
            hop_length   = hop_length
        )[0]
        mean_rms  = np.mean(rms)
        threshold = mean_rms + 2 * np.std(rms)

        gain = np.ones(len(rms))
        for i, r in enumerate(rms):
            if r > threshold:
                gain[i] = mean_rms / r

        gain_smooth = gaussian_filter1d(gain, sigma=5)
        gain_full   = np.interp(
            np.arange(len(audio_cleaned)),
            np.arange(len(rms)) * hop_length,
            gain_smooth
        )
        audio_cleaned = audio_cleaned * gain_full

        # Normalize volume
        max_amp = np.max(np.abs(audio_cleaned))
        if max_amp > 0:
            audio_cleaned = audio_cleaned / max_amp * 0.9

        print(f'Cleaning     : done')

    except Exception as e:
        print(f'Cleaning skipped ({e}) — using raw audio')
        audio_cleaned = audio

    # ── Step 3 — Split into 3 second chunks ──────────────
    chunk_size   = 3 * 16000
    total_chunks = len(audio_cleaned) // chunk_size

    if total_chunks == 0:
        padding       = chunk_size - len(audio_cleaned)
        audio_cleaned = np.pad(audio_cleaned, (0, padding), mode='constant')
        total_chunks  = 1

    print(f'Total chunks : {total_chunks} x 3 seconds')
    print(f'{"="*50}')

    chunk_results = []

    for i in range(total_chunks):
        start = i * chunk_size
        end   = start + chunk_size
        chunk = audio_cleaned[start:end]

        # ── Step 4 — Extract MFCC features ───────────────
        mfcc     = librosa.feature.mfcc(y=chunk, sr=16000, n_mfcc=40)
        features = np.mean(mfcc, axis=1).reshape(1, -1)

        # ── Step 5 — Normalize ────────────────────────────
        features_scaled = scaler.transform(features)

        # ── Step 6 — Predict ──────────────────────────────
        prediction = model.predict(features_scaled)[0]
        confidence = model.predict_proba(features_scaled)[0]
        conf_pct   = confidence[prediction] * 100
        label      = 'FAKE' if prediction == 1 else 'REAL'

        chunk_results.append({
            'chunk'      : i + 1,
            'start'      : i * 3,
            'end'        : (i + 1) * 3,
            'label'      : label,
            'confidence' : conf_pct
        })

        print(f'Chunk {i+1} [{i*3}s — {(i+1)*3}s] : {label} ({conf_pct:.1f}%)')

    # ── Step 7 — Majority vote ────────────────────────────
    fake_count = sum(1 for r in chunk_results if r['label'] == 'FAKE')
    real_count = sum(1 for r in chunk_results if r['label'] == 'REAL')
    avg_conf   = sum(r['confidence'] for r in chunk_results) / len(chunk_results)
    final      = 'FAKE' if fake_count > real_count else 'REAL'

    print(f'{"="*50}')
    print(f'Real chunks      : {real_count}')
    print(f'Fake chunks      : {fake_count}')
    print(f'Average confidence : {avg_conf:.1f}%')
    print(f'{"="*50}')
    print(f'FINAL RESULT : {final} '
          f'({"AI Generated Voice" if final == "FAKE" else "Human Voice"})')
    print(f'{"="*50}')

    return chunk_results, final, avg_conf

## Step 7 — generate_timeline_chart() Function
Creates a visual timeline showing which 3-second segments are Fake or Real.

In [24]:
def generate_timeline_chart(results, filename):
    '''Generate a horizontal timeline chart showing verdict for each 3-second chunk.'''
    fig, ax = plt.subplots(figsize=(10, 2.5))
    fig.patch.set_facecolor('white')

    for r in results:
        color     = FAKE_RED   if r['label'] == 'FAKE' else REAL_GREEN
        edgecolor = '#A32D2D'  if r['label'] == 'FAKE' else '#27500A'
        ax.barh(y=0, width=3, left=r['start'], color=color, alpha=0.85,
                height=0.6, edgecolor=edgecolor, linewidth=1.5)
        ax.text(r['start'] + 1.5,  0.18, r['label'],
                ha='center', va='center', fontsize=10, color='white', fontweight='bold')
        ax.text(r['start'] + 1.5, -0.15, f"{r['confidence']:.0f}% confident",
                ha='center', va='center', fontsize=9, color='white')

    total_duration = results[-1]['end']
    for r in results:
        ax.text(r['start'], -0.42, f"{r['start']}s",
                ha='center', fontsize=9, color=DARK_GRAY)
    ax.text(total_duration, -0.42, f"{total_duration}s",
            ha='center', fontsize=9, color=DARK_GRAY)

    fake_patch = mpatches.Patch(color=FAKE_RED,   label='FAKE — AI Generated')
    real_patch = mpatches.Patch(color=REAL_GREEN, label='REAL — Human Voice')
    ax.legend(handles=[fake_patch, real_patch], loc='upper right', fontsize=9, framealpha=0.9)

    ax.set_xlim(-0.2, total_duration + 0.2)
    ax.set_ylim(-0.55, 0.55)
    ax.set_yticks([])
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_title('Each block = 3 seconds of audio. Color shows the verdict for that segment.',
                 fontsize=9, color=GRAY, pad=8)

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_PATH, f'{filename}_{timestamp}_timeline.png')
    plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return chart_path

print('generate_timeline_chart() function defined.')

generate_timeline_chart() function defined.


## Step 8 — generate_analysis_chart() Function
Creates SHAP feature importance chart and normal range comparison chart.

In [25]:
def generate_analysis_chart(file_path, chunk_results, filename, final):
    '''Generate SHAP importance chart and normal range comparison chart.'''
    explainer = shap.TreeExplainer(model)

    fake_chunks = [r for r in chunk_results if r['label'] == 'FAKE']
    target      = fake_chunks[0] if fake_chunks else chunk_results[0]

    audio, sr = librosa.load(file_path, sr=16000, mono=True)
    start     = target['start'] * 16000
    chunk     = audio[start:start + (3 * 16000)]

    mfcc            = librosa.feature.mfcc(y=chunk, sr=16000, n_mfcc=40)
    features        = np.mean(mfcc, axis=1).reshape(1, -1)
    features_scaled = scaler.transform(features)

    shap_values = explainer.shap_values(features_scaled)
    if isinstance(shap_values, list):
        vals = np.array(shap_values[1][0]).flatten()
    elif hasattr(shap_values, 'values'):
        vals = np.array(shap_values.values).flatten()[:40]
    else:
        vals = np.array(shap_values).flatten()[:40]
    vals = vals[:40]

    feature_names = [f'MFCC_{i+1}' for i in range(40)]
    indices       = [int(i) for i in np.argsort(np.abs(vals))[-12:]]
    top_vals      = [vals[i] for i in indices]
    top_keys      = [feature_names[i] for i in indices]
    top_names     = [f"{MFCC_NAMES.get(k, k)}\n({k})" for k in top_keys]
    bar_colors    = [FAKE_RED if v > 0 else REAL_GREEN for v in top_vals]

    # Use pre-computed real voice statistics loaded from models/ folder
    norm_means = [real_means[i] for i in indices]
    norm_stds  = [real_stds[i]  for i in indices]
    norm_lo    = [m - s for m, s in zip(norm_means, norm_stds)]
    norm_hi    = [m + s for m, s in zip(norm_means, norm_stds)]

    fig = plt.figure(figsize=(16, 8))
    fig.patch.set_facecolor('white')
    gs  = gridspec.GridSpec(1, 2, figure=fig, wspace=0.45)

    # Left — SHAP importance
    ax1  = fig.add_subplot(gs[0])
    bars = ax1.barh(top_names, top_vals, color=bar_colors, alpha=0.85,
                    edgecolor='white', linewidth=0.5)
    ax1.axvline(x=0, color=DARK_GRAY, linewidth=1)
    for bar, val in zip(bars, top_vals):
        x = bar.get_width()
        ax1.text(x + (0.001 if x >= 0 else -0.001),
                 bar.get_y() + bar.get_height() / 2,
                 f'{"+" if val > 0 else ""}{val:.3f}',
                 va='center', ha='left' if x >= 0 else 'right',
                 fontsize=8, color=FAKE_RED if val > 0 else REAL_GREEN)
    ax1.set_xlabel('SHAP Value — how strongly this feature influences the decision',
                   fontsize=9, color=GRAY)
    title_word = 'FAKE' if final == 'FAKE' else 'REAL'
    ax1.set_title(f'Why is it {title_word}?\nTop contributing voice features',
                  fontsize=11, fontweight='bold', pad=12)
    ax1.tick_params(axis='y', labelsize=8)
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)
    ax1.legend(handles=[
        mpatches.Patch(color=FAKE_RED,   label='Pushing toward FAKE'),
        mpatches.Patch(color=REAL_GREEN, label='Pushing toward REAL')
    ], fontsize=8, loc='lower right')

    # Right — Normal range comparison
    ax2           = fig.add_subplot(gs[1])
    y_pos         = range(len(top_names))
    actual_vals   = [features_scaled[0][i] for i in indices]

    for j, (lo, hi) in enumerate(zip(norm_lo, norm_hi)):
        ax2.barh(j, hi - lo, left=lo, color=NORMAL_BLUE, alpha=0.35,
                 height=0.5, label='Normal range' if j == 0 else '')

    for j, (act, val) in enumerate(zip(actual_vals, top_vals)):
        inside     = norm_lo[j] <= act <= norm_hi[j]
        dot_color  = REAL_GREEN if inside else FAKE_RED
        ax2.scatter(act, j, color=dot_color, zorder=5, s=100,
                    edgecolors='white', linewidth=1)
        status       = 'Within range' if inside else 'Outside range'
        status_color = REAL_GREEN if inside else FAKE_RED
        xlim         = ax2.get_xlim()
        x_label      = xlim[1] if xlim[1] != 0.0 else 2
        ax2.text(x_label, j, f'  {status}', va='center',
                 fontsize=7.5, color=status_color)

    ax2.set_yticks(list(y_pos))
    ax2.set_yticklabels(top_names, fontsize=8)
    ax2.set_xlabel('Feature value (normalized). Blue = normal human voice range.',
                   fontsize=9, color=GRAY)
    ax2.set_title('Normal Range vs This Audio\nDot outside blue bar = suspicious',
                  fontsize=11, fontweight='bold', pad=12)
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)
    ax2.legend(handles=[
        mpatches.Patch(color=NORMAL_BLUE, alpha=0.5, label='Normal human voice range'),
        mpatches.Patch(color=REAL_GREEN,             label='Within normal range'),
        mpatches.Patch(color=FAKE_RED,               label='Outside normal range (suspicious)'),
    ], fontsize=8, loc='lower right')

    plt.tight_layout()
    chart_path = os.path.join(CHARTS_PATH, f'{filename}_{timestamp}_analysis.png')
    plt.savefig(chart_path, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    return chart_path

print('generate_analysis_chart() function defined.')

generate_analysis_chart() function defined.


## Step 9 — generate_pdf_report() Function
Builds the full PDF report with verdict banner, timeline chart, analysis charts, and chunk table.

In [26]:
def generate_pdf_report(file_path, results, final, avg_conf,
                        timeline_chart, analysis_chart):
    '''Generate a professional PDF report for the analyzed audio file.'''
    doc    = SimpleDocTemplate(
        REPORT_PATH, pagesize=A4,
        topMargin=0.5*inch, bottomMargin=0.5*inch,
        leftMargin=0.6*inch, rightMargin=0.6*inch
    )
    styles = getSampleStyleSheet()
    story  = []

    title_style = ParagraphStyle('CustomTitle', parent=styles['Title'],
                                 fontSize=22, spaceAfter=4,
                                 textColor=colors.HexColor(DARK_GRAY))
    sub_style   = ParagraphStyle('SubTitle', parent=styles['Normal'],
                                 fontSize=10, textColor=colors.HexColor(GRAY),
                                 spaceAfter=16)
    section_style = ParagraphStyle('SectionTitle', parent=styles['Heading2'],
                                   fontSize=13, textColor=colors.HexColor(DARK_GRAY),
                                   spaceBefore=14, spaceAfter=6, borderPad=4)
    explain_style = ParagraphStyle('Explain', parent=styles['Normal'],
                                   fontSize=9, textColor=colors.HexColor(GRAY),
                                   backColor=colors.HexColor('#F1EFE8'),
                                   borderPad=8, spaceAfter=10,
                                   leftIndent=8, rightIndent=8)
    normal_style  = ParagraphStyle('Normal2', parent=styles['Normal'],
                                   fontSize=10, spaceAfter=6)

    story.append(Paragraph('Deepfake Voice Detection Report', title_style))
    story.append(Paragraph(
        f'Pakistani Urdu-English Mixed Audio Analysis · '
        f'Generated {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}',
        sub_style
    ))
    story.append(HRFlowable(width='100%', thickness=1,
                             color=colors.HexColor('#D3D1C7')))
    story.append(Spacer(1, 0.15*inch))

    if final == 'FAKE':
        verdict_text  = 'AI Generated Voice Detected'
        verdict_sub   = 'This audio is likely synthetic — not a real human voice'
        banner_bg     = colors.HexColor(FAKE_LIGHT)
        banner_border = colors.HexColor(FAKE_RED)
        verdict_color = colors.HexColor('#791F1F')
        conf_bg       = colors.HexColor('#F09595')
    else:
        verdict_text  = 'Real Human Voice Confirmed'
        verdict_sub   = 'This audio appears to be a genuine human voice'
        banner_bg     = colors.HexColor(REAL_LIGHT)
        banner_border = colors.HexColor(REAL_GREEN)
        verdict_color = colors.HexColor('#27500A')
        conf_bg       = colors.HexColor('#C0DD97')

    fake_count = sum(1 for r in results if r['label'] == 'FAKE')
    real_count = sum(1 for r in results if r['label'] == 'REAL')

    verdict_data = [[
        Paragraph(f'<b>{verdict_text}</b>',
                  ParagraphStyle('VT', fontSize=16, textColor=verdict_color, spaceAfter=4)),
        Paragraph(f'<b>{avg_conf:.1f}%</b><br/>Confidence',
                  ParagraphStyle('VC', fontSize=14, textColor=verdict_color, alignment=TA_CENTER))
    ],[
        Paragraph(verdict_sub,
                  ParagraphStyle('VS', fontSize=9,
                                 textColor=colors.HexColor('#3B6D11' if final=='REAL' else '#A32D2D'))),
        Paragraph(f'File: {os.path.basename(file_path)}',
                  ParagraphStyle('VF', fontSize=8, textColor=colors.HexColor(GRAY),
                                 alignment=TA_CENTER))
    ]]
    verdict_table = Table(verdict_data, colWidths=[4.5*inch, 1.8*inch])
    verdict_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), banner_bg),
        ('LINEABOVE',  (0,0), (-1,0),  4, banner_border),
        ('ROUNDEDCORNERS', [8]),
        ('VALIGN',     (0,0), (-1,-1), 'MIDDLE'),
        ('PADDING',    (0,0), (-1,-1), 12),
        ('BACKGROUND', (1,0), (1,-1),  conf_bg),
    ]))
    story.append(verdict_table)
    story.append(Spacer(1, 0.15*inch))

    duration   = results[-1]['end']
    stats_data = [[
        Paragraph('<b>Total Segments</b><br/>' + str(len(results)),
                  ParagraphStyle('S',  fontSize=11, alignment=TA_CENTER,
                                 textColor=colors.HexColor(DARK_GRAY))),
        Paragraph(f'<b>Fake Segments</b><br/><font color="{FAKE_RED}">{fake_count}</font>',
                  ParagraphStyle('S2', fontSize=11, alignment=TA_CENTER,
                                 textColor=colors.HexColor(DARK_GRAY))),
        Paragraph(f'<b>Real Segments</b><br/><font color="{REAL_GREEN}">{real_count}</font>',
                  ParagraphStyle('S3', fontSize=11, alignment=TA_CENTER,
                                 textColor=colors.HexColor(DARK_GRAY))),
        Paragraph(f'<b>Audio Duration</b><br/>{duration}s',
                  ParagraphStyle('S4', fontSize=11, alignment=TA_CENTER,
                                 textColor=colors.HexColor(DARK_GRAY))),
    ]]
    stats_table = Table(stats_data, colWidths=[1.575*inch]*4)
    stats_table.setStyle(TableStyle([
        ('BACKGROUND', (0,0), (-1,-1), colors.HexColor('#F1EFE8')),
        ('GRID',       (0,0), (-1,-1), 0.5, colors.HexColor('#D3D1C7')),
        ('PADDING',    (0,0), (-1,-1), 10),
        ('ALIGN',      (0,0), (-1,-1), 'CENTER'),
        ('VALIGN',     (0,0), (-1,-1), 'MIDDLE'),
    ]))
    story.append(stats_table)
    story.append(Spacer(1, 0.1*inch))

    story.append(Paragraph('Timeline — Which Seconds Are Suspicious?', section_style))
    story.append(Paragraph(
        'Each colored block represents 3 seconds of audio. '
        'Red = AI Generated (Fake). Green = Human Voice (Real). '
        'The percentage inside shows how confident the model is for that segment.',
        explain_style
    ))
    story.append(Image(timeline_chart, width=6.3*inch, height=1.8*inch))
    story.append(Spacer(1, 0.1*inch))

    title_word = 'FAKE' if final == 'FAKE' else 'REAL'
    story.append(Paragraph(f'Voice Feature Analysis — Why is it {title_word}?', section_style))
    story.append(Paragraph(
        'LEFT CHART: Red bars = features pushing toward FAKE. '
        'Green bars = pushing toward REAL. Longer bar = stronger influence. '
        'RIGHT CHART: Blue bar = normal range of real human voices. '
        'Colored dot = where this audio falls. '
        'Dot outside blue bar = that feature is abnormal.',
        explain_style
    ))
    story.append(Image(analysis_chart, width=6.3*inch, height=4.2*inch))
    story.append(Spacer(1, 0.1*inch))

    story.append(Paragraph('Segment-by-Segment Breakdown', section_style))
    story.append(Paragraph(
        'Detailed results for each 3-second segment of the audio file.',
        explain_style
    ))

    chunk_data = [[
        Paragraph('<b>Segment</b>',        styles['Normal']),
        Paragraph('<b>Time</b>',           styles['Normal']),
        Paragraph('<b>Result</b>',         styles['Normal']),
        Paragraph('<b>Confidence</b>',     styles['Normal']),
        Paragraph('<b>Suspicion Level</b>',styles['Normal']),
    ]]
    for r in results:
        is_fake   = r['label'] == 'FAKE'
        res_color = FAKE_RED   if is_fake else REAL_GREEN
        bar_filled = int(r['confidence'] / 100 * 20)
        bar_str    = '█' * bar_filled + '░' * (20 - bar_filled)
        chunk_data.append([
            Paragraph(str(r['chunk']), styles['Normal']),
            Paragraph(f"{r['start']}s — {r['end']}s", styles['Normal']),
            Paragraph(f'<font color="{res_color}"><b>{r["label"]}</b></font>',
                      styles['Normal']),
            Paragraph(f"{r['confidence']:.1f}%", styles['Normal']),
            Paragraph(f'<font color="{res_color}" size="7">{bar_str}</font>',
                      styles['Normal']),
        ])

    chunk_table = Table(chunk_data,
                        colWidths=[0.7*inch, 1.1*inch, 0.9*inch, 1.0*inch, 2.6*inch])
    chunk_table.setStyle(TableStyle([
        ('BACKGROUND',    (0,0), (-1,0),  colors.HexColor(DARK_GRAY)),
        ('TEXTCOLOR',     (0,0), (-1,0),  colors.white),
        ('FONTNAME',      (0,0), (-1,0),  'Helvetica-Bold'),
        ('FONTSIZE',      (0,0), (-1,0),  10),
        ('ROWBACKGROUNDS',(0,1), (-1,-1),
         [colors.white, colors.HexColor('#F1EFE8')]),
        ('GRID',          (0,0), (-1,-1), 0.5, colors.HexColor('#D3D1C7')),
        ('FONTSIZE',      (0,1), (-1,-1), 10),
        ('PADDING',       (0,0), (-1,-1), 8),
        ('VALIGN',        (0,0), (-1,-1), 'MIDDLE'),
    ]))
    story.append(chunk_table)
    story.append(Spacer(1, 0.3*inch))

    story.append(HRFlowable(width='100%', thickness=0.5,
                             color=colors.HexColor('#D3D1C7')))
    story.append(Spacer(1, 0.1*inch))
    story.append(Paragraph(
        'Generated by Deepfake Voice Detection System — '
        'Pakistani Urdu-English Code-Switched Audio Analysis | '
        'Model Accuracy: ~95% | Features: MFCC (40) | '
        'Classifier: Random Forest with class_weight=balanced',
        ParagraphStyle('Footer', fontSize=8, textColor=colors.HexColor(GRAY),
                       alignment=TA_CENTER)
    ))

    doc.build(story)

print('generate_pdf_report() function defined.')

generate_pdf_report() function defined.


## Step 10 — Run on a Test File
Change TEST_FILE to any .wav file you want to analyze.

In [ ]:
# Change this path to any .wav file you want to analyze
TEST_FILE   = os.path.join(BASE_DIR, 'dataset', 'real', 'real_001.wav')
# TEST_FILE = r"D:\Hamza\NIAI\NIAI_Project\DeepfakeProject\dataset\real\real_165.wav"
audio_name  = os.path.splitext(os.path.basename(TEST_FILE))[0]
timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
REPORT_PATH = os.path.join(REPORTS_FOLDER, f'{audio_name}_{timestamp}.pdf')

print(f'Test file : {TEST_FILE}')
print(f'Report    : {REPORT_PATH}')
print()

results, final, avg_conf = predict_audio(TEST_FILE)

print('\nGenerating timeline chart...')
timeline_chart = generate_timeline_chart(results, audio_name)
print('Timeline chart done')

print('Generating analysis chart...')
analysis_chart = generate_analysis_chart(TEST_FILE, results, audio_name, final)
print('Analysis chart done')

print('Generating PDF report...')
generate_pdf_report(TEST_FILE, results, final, avg_conf, timeline_chart, analysis_chart)
print('PDF report saved!')

print(f'\n{"="*55}')
print(f'Report : audio_reports/{audio_name}_{timestamp}.pdf')
print(f'Charts : audio_reports/charts/')
print(f'{"="*55}')

Test file : D:\Hamza\NIAI\NIAI_Project\DeepfakeProject\dataset\real\real_165.wav
Report    : d:\Hamza\NIAI\NIAI_Project\deepfake-voice-detection\audio_reports\real_165_20260531_190327.pdf

Analyzing : real_165.wav
Duration     : 10.0 seconds
Cleaning     : done
Total chunks : 3 x 3 seconds
Chunk 1 [0s — 3s] : REAL (78.0%)
Chunk 2 [3s — 6s] : REAL (75.0%)
Chunk 3 [6s — 9s] : REAL (64.0%)
Real chunks      : 3
Fake chunks      : 0
Average confidence : 72.3%
FINAL RESULT : REAL (Human Voice)

Generating timeline chart...
Timeline chart done
Generating analysis chart...


C:\Users\HP\AppData\Local\Temp\ipykernel_8168\3936763576.py:102: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Analysis chart done
Generating PDF report...
PDF report saved!

Report : audio_reports/real_165_20260531_190327.pdf
Charts : audio_reports/charts/
